# The Google ADK Agent Tree

**What this notebook shows**: the complete Google ADK (`google-adk >= 2.7.1`) agent tree for the gemini-hackathon submission. Walks the LlmAgent + 5 tools + App + Runner + Fleet primitives stack.

**Source**: derived from the `gemini_hackathon/agents/adk_gemini_agent.py` (Phase 1-3 lift + extensions).

**5-layer walkthrough**:
1. (this file) Title + provenance
2. The LlmAgent + 5 FunctionTool + App + Runner stack
3. The AGUI 13-event protocol (see `agui_event_protocol.ipynb`)
4. The CopilotKit consumption (see `copilotkit_runtime_config.ipynb`)
5. (This file) The run_agent_turn() pipeline


## Layer 2 — The LlmAgent + 5 FunctionTool + App + Runner stack

The `build_adk_agent()` factory in `gemini_hackathon/agents/adk_gemini_agent.py`:
- Wraps the LlmAgent in `App(root_agent=..., name="gemini_hackathon")`
- Uses `Gemini(model="gemini-3.5-flash", retry_options=HttpRetryOptions(attempts=3))`
- Wires 5 FunctionTool wrappers (lookup_outcome, retrieve_resources, find_similar_resources, retrieve_safeguarding, mark_answer)
- Instantiates `InMemoryRunner(agent=app)`

Run the cell below to inspect the live agent tree.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

# Build the canonical ADK agent tree
from gemini_hackathon.agents.adk_gemini_agent import (
    GEMINI_HACKATHON_AGENT, build_adk_agent, _build_adk_tool_wrappers,
)

print("AGUI events:", len(GEMINI_HACKATHON_AGENT.tool_count if hasattr(GEMINI_HACKATHON_AGENT, 'tool_count') else 5))
print("Model:", GEMINI_HACKATHON_AGENT.model)
print("Tool descriptions:")
for tool in GEMINI_HACKATHON_AGENT.tools:
    print(f"  - {tool.name}: {tool.description[:80]}...")


## Layer 5 — The `run_agent_turn()` pipeline

The `run_agent_turn()` function (in `adk_gemini_agent.py`) composes:

1. `ModelArmor.check_prompt(message)` — input sanitisation (prompt-injection / PII guard)
2. `Observability.trace(agent_name, user_id, session_id, subnation)` — opens a Fleet trace
3. `runner.run(user_id, session_id, new_message=content)` — the real ADK invocation
4. `Observability.record_invocation(trace, agent_name, event_count, status)` — emits the cost + tokens event
5. `render_agui_events(raw_events)` — converts ADK Event → AgUiEvent for the AGUI stream

Run the cell below to see the Fleet-wrapped turn in action.


In [ ]:
from gemini_hackathon.agents.adk_gemini_agent import run_agent_turn, AgentTurnResult

result = run_agent_turn(
    message="What are the JC Maths outcomes for Ireland?",
    subnation="ireland",
    role="student",
    cycle="junior_cycle",
)
print(f"status: {result.status}")
print(f"events: {len(result.events)}")
print(f"model_armor.blocked: {result.model_armor_check.blocked if result.model_armor_check else 'n/a'}")
print(f"observability.trace_id: {result.observability.trace_id if result.observability else 'n/a'}")
